In [ ]:
from dotenv import load_dotenv
load_dotenv()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
quant_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", quantization_config=quant_config, device_map="auto")

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 339/339 [00:35<00:00,  9.61it/s] 


In [31]:
text = "His arms are sweaty, knees weak, arms are heavy. There's vommit on his sweater already, mom's spaghetti. He's nervous but on the surface he looks calm and ready to drop bombs, but he keeps on forgetting"

input = tokenizer(text, return_tensors="pt").to("cuda")
input_ids = input["input_ids"]
print(input_ids)

tensor([[15986, 11715,   525, 97421,    11, 30524,  7469,    11, 11715,   525,
          8811,    13,  2619,   594, 21982,  1763,   389,   806, 60121,  2669,
            11,  3368,   594, 86910,    13,  1260,   594, 22596,   714,   389,
           279,  7329,   566,  5868, 19300,   323,  5527,   311,  5943, 32506,
            11,   714,   566, 13598,   389, 65027]], device='cuda:0')


In [38]:
output = model.generate(input_ids)
new_tokens = output[0][input_ids.shape[-1]:]

/workspace/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1551: UserWarning: Using the model-agnostic default `max_length` (=66) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [39]:
new_text = tokenizer.decode(new_tokens)
print(new_text)

 words, and his palms are sweating and his brain is a wreck, and all he wants to do


In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [60]:
embed = embedder.encode(text)
print(embed)

[ 3.26018967e-02  5.07858302e-03  4.19486091e-02  4.29097824e-02
 -7.71104917e-02 -4.84629795e-02  7.75597990e-02 -6.28195284e-03
  1.05281025e-02  2.09310763e-02  4.16068435e-02 -6.29075691e-02
  2.48346385e-02 -2.24615559e-02 -6.39206693e-02 -2.05033291e-02
  6.22154884e-02 -3.99041437e-02 -5.09399325e-02  3.01495194e-02
 -9.20273596e-04  9.48430225e-02  1.19222021e-02 -3.95227410e-02
  2.12724432e-02  1.59614775e-02  4.56457399e-02  1.51490374e-02
 -7.21600130e-02 -3.98825668e-02 -8.25384855e-02  1.75082516e-02
  2.63091978e-02 -6.96767494e-03 -6.77549541e-02 -3.83729525e-02
  1.50698394e-01 -3.47482413e-02  1.98589172e-02 -4.46559601e-02
  6.12461241e-03  1.03815220e-01  5.01938909e-02  3.55488732e-02
 -6.02163225e-02  3.43557708e-02  9.32271034e-03 -5.56024760e-02
  2.21760403e-02  1.25288137e-03  1.32043222e-02 -1.79220531e-02
  7.18120160e-03 -3.33491899e-02  3.53802107e-02 -1.86446141e-02
  1.30649144e-02 -1.49654932e-02  2.95656193e-02  5.92753291e-02
  4.01429236e-02 -1.27953

In [50]:
import chromadb as chroma
client = chroma.EphemeralClient()

text = "His arms are sweaty, knees weak, arms are heavy. There's vommit on his sweater already, mom's spaghetti. He's nervous but on the surface he looks calm and ready to drop bombs, but he keeps on forgetting"

In [61]:
collection = client.get_or_create_collection(name="test_collection")

collection.upsert(
    ids="124",
    embeddings=embed,
    documents=text,
    metadatas=[{"timestamp": "12:01AM", "project": "test_proj"}]
)

In [59]:
text = "apples are nasty"

In [67]:
qa = "I'm nervous"
embed = embedder.encode(qa)

In [ ]:
results = collection.query(query_embeddings=[embed], n_results,include=["documents", "metadatas", "distances"])

In [70]:
results

{'ids': [['123', '124']],
 'embeddings': None,
 'documents': [["His arms are sweaty, knees weak, arms are heavy. There's vommit on his sweater already, mom's spaghetti. He's nervous but on the surface he looks calm and ready to drop bombs, but he keeps on forgetting",
   'apples are nasty']],
 'uris': None,
 'included': ['documents', 'metadatas', 'distances'],
 'data': None,
 'metadatas': [[{'timestamp': '12:00AM', 'project': 'test_proj'},
   {'timestamp': '12:01AM', 'project': 'test_proj'}]],
 'distances': [[1.3787699937820435, 1.6617240905761719]]}